In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed([
    'transformers',
    'datasets',
    'torch',
    'scikit-learn',
    'pandas',
    'numpy'
])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    torch_device = 'mps'
    pipeline_device = torch.device('mps')
else:
    torch_device = 'cpu'
    pipeline_device = -1

print({'selected_device': torch_device, 'pipeline_device': str(pipeline_device)})


In [ ]:
dataset = load_dataset('dair-ai/emotion', split='test')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({'dataset': 'dair-ai/emotion', 'split': 'test', 'num_rows': len(dataset)})
print(dataset[:3])


In [ ]:
model_name = 'j-hartmann/emotion-english-distilroberta-base'

classifier = pipeline(
    task='text-classification',
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True
)

model_to_dataset_label = {
    'sadness': 'sadness',
    'joy': 'joy',
    'love': 'love',
    'anger': 'anger',
    'fear': 'fear',
    'surprise': 'surprise'
}

label_to_id = {name: i for i, name in enumerate(class_names)}

print({'model_name': model_name, 'mapped_labels': model_to_dataset_label})


In [ ]:
texts = dataset['text']
true_ids = dataset['label']
true_labels = [class_names[i] for i in true_ids]

raw_outputs = classifier(texts, batch_size=64, truncation=True)

pred_labels = []
confidence_scores = []

for item in raw_outputs:
    model_label = item['label'].lower().strip()
    mapped_label = model_to_dataset_label.get(model_label)
    if mapped_label is None:
        raise ValueError(f'Unmapped model label: {item["label"]}')
    pred_labels.append(mapped_label)
    confidence_scores.append(float(item['score']))

pred_ids = [label_to_id[label] for label in pred_labels]

results_df = pd.DataFrame({
    'text': texts,
    'true_label': true_labels,
    'predicted_label': pred_labels,
    'confidence_score': confidence_scores
})

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'test',
    'num_examples': len(dataset),
    'device': torch_device,
    'accuracy': round(float(accuracy), 6)
})
print(report)


In [ ]:
sample_n = 8
print(results_df.head(sample_n).to_string(index=False))
